# Diagnostic 1 (merger 1s) — score vs kernel alignment, per event (interactive)

Same diagnostic as the pre-merger notebook, for the **merger 1 s** model (63-64s, kernel 1 s, `window_offset = 0`, so trained alignment is `e = 0`).

Each line is **one event**: the model's score as the kernel slides across the merger.

- x-axis `e` = (kernel right-edge time) − (coalescence). `e = 0` → the 1 s kernel ends at the merger (**the trained point**); `e < 0` → kernel entirely pre-merger; `e > 0` → kernel after the merger.
- **red = signal** (a loud injection) — should **dip in σ / peak in −σ around `e ≈ 0`**. With only a 1 s kernel the bump is narrow (the signal is in-band for a short window).
- **blue = background** (no signal) — should stay **flat and uncertain**.

**Hover a line to highlight it** (the rest dim out). Click legend entries to show/hide.

Generate the CSV first with `diagnostics/diag_score_timeseries.py` (needs a GPU). This notebook is pure plotting.

In [ ]:
import pandas as pd
import plotly.graph_objects as go

CSV = "/n/holystore01/LABS/iaifi_lab/Lab/kyoon/aframe_linoss/runs/regression_sv/diag/merger_63-64s_ft/score_timeseries.csv"
Y_COL = "neg_sigma"   # what to plot on y: 'neg_sigma' (detection statistic), 'chirp_sigma', or 'chirp_mean'
TRAINED_E = 0.0       # merger model: kernel ends at coalescence -> trained alignment e = 0

df = pd.read_csv(CSV)
print(f"{df.event_id.nunique()} events, {len(df)} points")
df.head()

In [ ]:
SIGNAL_RGB, BG_RGB = "214,39,40", "31,119,180"   # red, blue
rgb_of = {}   # trace index -> 'r,g,b' so we can restyle alpha on hover

fig = go.FigureWidget()
for eid, g in df.groupby("event_id"):
    g = g.sort_values("e")
    is_signal = g["kind"].iloc[0] == "signal"
    rgb = SIGNAL_RGB if is_signal else BG_RGB
    snr = g["snr"].iloc[0]
    name = f"{eid} (SNR {snr:.0f})" if is_signal else eid
    fig.add_scatter(
        x=g["e"], y=g[Y_COL], mode="lines+markers", name=name,
        legendgroup=g["kind"].iloc[0],
        line=dict(color=f"rgba({rgb},0.35)", width=1.5),
        marker=dict(size=4, color=f"rgba({rgb},0.35)"),
    )
    rgb_of[len(fig.data) - 1] = rgb

fig.add_vline(x=TRAINED_E, line=dict(color="green", dash="dash"),
              annotation_text="trained = merger (e=0)", annotation_position="top")
fig.update_layout(
    template="plotly_white", height=620, hovermode="closest",
    xaxis_title="e = kernel right-edge − coalescence [s]   (0 = kernel ends at merger)",
    yaxis_title=Y_COL,
    title="Merger 1s — score vs kernel alignment (hover a line to highlight)",
)

def _style(i, alpha, width):
    col = f"rgba({rgb_of[i]},{alpha})"
    fig.data[i].line.width = width
    fig.data[i].line.color = col
    fig.data[i].marker.color = col

def _highlight(trace, points, state):
    h = points.trace_index
    with fig.batch_update():
        for i in rgb_of:
            _style(i, 1.0, 4) if i == h else _style(i, 0.07, 1)

def _reset(trace, points, state):
    with fig.batch_update():
        for i in rgb_of:
            _style(i, 0.35, 1.5)

for tr in fig.data:
    if hasattr(tr, "on_hover"):
        tr.on_hover(_highlight)
        tr.on_unhover(_reset)
fig

**Reading it:** the red (signal) `−σ` should rise above flat blue (background) in a **narrow** bump near `e = 0` (the 1 s kernel only sees the signal for a short alignment window). Compared with the merger 4 s model, expect a sharper but possibly lower peak (less signal captured in 1 s). If red and blue overlap everywhere, the model isn't separating signal from noise at that SNR — lower `--snr-min`, or set `Y_COL = "chirp_sigma"` / `"chirp_mean"`.